<a href="https://colab.research.google.com/github/shyamsundarsah123/machine-learning/blob/main/lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#A1- ChatGPT

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import time
import numpy as np
import pandas as pd

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

#A1 - ChatGPT

BASE_PATH = "/content/drive/MyDrive/ML_lab"
IMAGE_ROOT = os.path.join(BASE_PATH, "BreaKHis_v1")

CSV_PATH = os.path.join(
    BASE_PATH,
    "BreaKHis_metadata.csv"
)

SAMPLE_COUNT = 100
RESIZE_SHAPE = (16, 16)

DEFAULT_K = 3
DISTANCE = "euclidean"
SORT_METHOD = "insertion"

SEED = 42

In [ ]:
#A1/1 - ChatGPT

def locate_image(relative_name):
    relative_name = str(relative_name).replace("\", "/")

    candidates = [
        os.path.join(BASE_PATH, relative_name),
        os.path.join(IMAGE_ROOT, relative_name)
    ]

    if relative_name.startswith("BreaKHis_v1/"):
        short_name = relative_name.split(
            "BreaKHis_v1/", 1
        )[1]

        candidates.append(
            os.path.join(IMAGE_ROOT, short_name)
        )

    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate

    return None


def get_target(path):
    path = str(path).lower()

    if "/benign/" in path:
        return "benign"

    if "/malignant/" in path:
        return "malignant"

    return None


def image_to_vector(path, size=(16, 16)):
    image = Image.open(path).convert("L")
    image = image.resize(size)

    values = np.asarray(
        image,
        dtype=np.float32
    )

    return values.reshape(-1)


def create_image_features(metadata, per_class=100):
    metadata = metadata.copy()

    metadata["target"] = metadata["filename"].map(
        get_target
    )

    metadata = metadata.dropna(
        subset=["target"]
    )

    selected_parts = []

    for target in ["benign", "malignant"]:

        class_rows = metadata[
            metadata["target"] == target
        ]

        selected_parts.append(
            class_rows.sample(
                n=min(per_class, len(class_rows)),
                random_state=SEED
            )
        )

    selected = pd.concat(
        selected_parts,
        ignore_index=True
    )

    features = []
    targets = []

    for _, record in selected.iterrows():

        image_path = locate_image(
            record["filename"]
        )

        if image_path is None:
            continue

        try:
            vector = image_to_vector(
                image_path,
                RESIZE_SHAPE
            )

            features.append(vector)
            targets.append(record["target"])

        except Exception:
            continue

    return (
        np.asarray(features, dtype=np.float32),
        np.asarray(targets)
    )
